In [6]:
import os
import json

# --- CONFIGURATION (ADJUST THESE PATHS) ---
# Directory containing your 'annotations', 'annotations2', etc., folders
BLIP_CAPTIONS_PARENT_DIR = "/home/poorna/data/blip_dir" 
# The final consolidated file path
BLIP_CONSOLIDATED_FILE = "/home/poorna/data/consolidate_blip_captions.json" 
FILE_EXTENSION = ".txt" # <-- Confirmed based on your structure
# -------------------------------------------

def consolidate_blip_captions_final(parent_dir, output_file, extension):
    """
    Reads all captions from the specified directory structure, using the .txt extension.
    """
    all_blip_captions = []
    
    # Generate the expected list of directories (annotations1, annotations2, ..., annotations7)
    # and the corresponding base file names (ann1, ann2, ..., ann7)
    expected_structure = [
        (f'annotations{i}', f'ann{i}') for i in range(1, 8)
    ]
    
    print(f"Consolidating files with extension '{extension}' based on structure: {expected_structure}")

    for dir_name, base_file_name in expected_structure:
        dir_path = os.path.join(parent_dir, dir_name)
        
        # --- FIX: Dynamically add the confirmed .txt extension ---
        file_path = os.path.join(dir_path, base_file_name + extension) 
        
        if os.path.exists(file_path):
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    # Read lines, strip whitespace/newlines, and filter out any empty lines
                    captions_in_file = [line.strip() for line in f if line.strip()]
                    all_blip_captions.extend(captions_in_file)
                    print(f"Loaded {len(captions_in_file)} captions from {dir_name}/{base_file_name}{extension}")
            except Exception as e:
                print(f"Error reading file {file_path}: {e}")
        else:
            print(f"Warning: Caption file not found at {file_path}. Skipping directory.")

    print(f"\nTotal BLIP captions loaded: {len(all_blip_captions)}")

    # Save the consolidated list to a single JSON file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_blip_captions, f)
        
    print(f"Consolidated captions saved to: {output_file}")
    return output_file
    
# Run consolidation with the correct function name and arguments
consolidated_path = consolidate_blip_captions_final(
    BLIP_CAPTIONS_PARENT_DIR,  
    BLIP_CONSOLIDATED_FILE,
    FILE_EXTENSION 
)

print("-" * 50)
print("CONSOLIDATION COMPLETE. You can now run the HDF5 generation script.")

Consolidating files with extension '.txt' based on structure: [('annotations1', 'ann1'), ('annotations2', 'ann2'), ('annotations3', 'ann3'), ('annotations4', 'ann4'), ('annotations5', 'ann5'), ('annotations6', 'ann6'), ('annotations7', 'ann7')]
Loaded 200 captions from annotations1/ann1.txt
Loaded 200 captions from annotations2/ann2.txt
Loaded 200 captions from annotations3/ann3.txt
Loaded 200 captions from annotations4/ann4.txt
Loaded 200 captions from annotations5/ann5.txt
Loaded 200 captions from annotations6/ann6.txt
Loaded 200 captions from annotations7/ann7.txt

Total BLIP captions loaded: 1400
Consolidated captions saved to: /home/poorna/data/consolidate_blip_captions.json
--------------------------------------------------
CONSOLIDATION COMPLETE. You can now run the HDF5 generation script.


In [20]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
import h5py
from tqdm.auto import tqdm

# --- 1. MINIMAL DATASET CLASS (EEG + BLIP ONLY) ---

class EEGBliPDataset(Dataset): # Renamed class for clarity
    """
    Minimal Dataset: Loads only EEG and BLIP Caption IDs.
    All other metadata and original captions are ignored.
    """
    def __init__(self, eeg_dir, tokenizer, blip_captions_file, max_length=64):
        
        self.tokenizer = tokenizer
        self.max_length = max_length

        # -------------------------
        # 1. Load EEG files (all subjects)
        # -------------------------
        eeg_files = []
        for root, dirs, files in os.walk(eeg_dir):
            for f in files:
                if f.endswith(".npy") and "_preprocessed" in f:
                    eeg_files.append(os.path.join(root, f))
        eeg_files = sorted(eeg_files)

        if not eeg_files:
            raise FileNotFoundError(f"No EEG .npy files found in {eeg_dir}")

        self.eeg_file_paths = eeg_files
        self.eeg_data_list = []
        self.index_map = []

        for subj_idx, path in enumerate(self.eeg_file_paths):
            eeg = np.load(path, mmap_mode='r')
            assert eeg.ndim == 3 and eeg.shape[1:] == (62, 400), \
                f"EEG file {path} has shape {eeg.shape}, expected (*, 62, 400)"
            self.eeg_data_list.append(eeg)
            n_samples = eeg.shape[0]
            self.index_map.extend([(subj_idx, i) for i in range(n_samples)])

        total_samples = len(self.index_map)
        num_subjects = len(self.eeg_file_paths)
        
        print(f"Found {num_subjects} EEG files → Total samples: {total_samples}")

        if num_subjects == 0:
             raise ValueError("No EEG files were loaded.")
             
        num_unique_stimuli = total_samples // num_subjects
        print(f"Calculated number of unique stimuli: {num_unique_stimuli}")

        # -------------------------
        # 2. Load BLIP Captions 
        # -------------------------
        
        print(f"Loading BLIP captions from {blip_captions_file}...")
        try:
            with open(blip_captions_file, 'r', encoding='utf-8') as f:
                base_blip_captions = json.load(f)
        except FileNotFoundError:
            raise FileNotFoundError(f"BLIP captions file not found at {blip_captions_file}.")

        if len(base_blip_captions) != num_unique_stimuli:
             raise ValueError(
                 f"Data Mismatch: Loaded {len(base_blip_captions)} BLIP captions but expected {num_unique_stimuli} based on EEG trials. "
                 f"The BLIP file count must be correct (e.g., 1400)."
             )

        # Repeat captions for each subject
        self.blip_captions = base_blip_captions * num_subjects
        
        print(f"Total BLIP captions repeated for {num_subjects} subjects.")

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        subj_idx, local_idx = self.index_map[idx]
        eeg_tensor = torch.tensor(self.eeg_data_list[subj_idx][local_idx], dtype=torch.float32)
        
        # 1. BLIP Caption Tokenization
        blip_caption = self.blip_captions[idx]
        if not isinstance(blip_caption, str): blip_caption = "" 
        blip_tokenized = self.tokenizer(
            blip_caption, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt"
        )
        blip_input_ids = blip_tokenized['input_ids'].squeeze(0) 
        
        # RETURN ONLY EEG and BLIP CAPTIONS (2 TENSORS)
        return eeg_tensor, blip_input_ids

# --- 2. HDF5 GENERATION SCRIPT (EEG + BLIP Only) ---

# --- Paths (Set by user) ---
HDF5_FILE = "/home/poorna/data/eeg_blip_only_final.h5" # <-- NEW OUTPUT FILENAME
EEG_DIR = "/home/poorna/data/preprocessed_eeg"
TOKENIZER_PATH = "/home/poorna/models/bert-base-uncased"

# Path to consolidated BLIP captions (from consolidation script)
BLIP_CONSOLIDATED_FILE = "/home/poorna/data/consolidate_blip_captions.json" 

# --- Dataset Initialization ---
print("\nInitializing dataset...")
tokenizer = BertTokenizer.from_pretrained(TOKENIZER_PATH)

dataset = EEGBliPDataset( # Renamed class
    eeg_dir=EEG_DIR,
    tokenizer=tokenizer,
    blip_captions_file=BLIP_CONSOLIDATED_FILE
)

loader = DataLoader(dataset, batch_size=1, shuffle=False)

# --- Get dynamic shapes from a sample ---
n_samples = len(dataset)

if n_samples == 0:
    print("Error: Dataset is empty.")
else:
    # Get 2 items from the dataset
    sample_eeg, sample_blip_input_ids = dataset[0]

    # --- Create HDF5 file ---
    print(f"\nCreating NEW HDF5 file at {HDF5_FILE}...")
    with h5py.File(HDF5_FILE, "w") as f:
        eeg_shape = (n_samples, *sample_eeg.shape)
        blip_token_shape = (n_samples, *sample_blip_input_ids.shape) 

        print(f"Allocating space for {n_samples} samples...")
        print(f"  - EEG shape: {eeg_shape}")
        print(f"  - BLIP Token shape: {blip_token_shape}") 

        eeg_ds = f.create_dataset("eeg", shape=eeg_shape, dtype="float32")
        blip_tokens_ds = f.create_dataset("blip_input_ids", shape=blip_token_shape, dtype="int64") 

        # Iterate and save
        print("Writing data to HDF5 file...")
        try:
            # Unpack 2 tensors from the loader
            for idx, (eeg_tensor, blip_input_ids_tensor) in enumerate(tqdm(loader, desc="Saving to HDF5")):
                eeg_ds[idx] = eeg_tensor.squeeze(0).numpy()
                blip_tokens_ds[idx] = blip_input_ids_tensor.squeeze(0).numpy() 
            
            print(f"\nSuccessfully saved new dataset to {HDF5_FILE}")

        except Exception as e:
            print(f"\n--- ERROR during HDF5 writing at index {idx} ---")
            print(f"Error: {e}")


Initializing dataset...
Found 20 EEG files → Total samples: 28000
Calculated number of unique stimuli: 1400
Loading BLIP captions from /home/poorna/data/consolidate_blip_captions.json...
Total BLIP captions repeated for 20 subjects.

Creating NEW HDF5 file at /home/poorna/data/eeg_blip_only_final.h5...
Allocating space for 28000 samples...
  - EEG shape: (28000, 62, 400)
  - BLIP Token shape: (28000, 64)
Writing data to HDF5 file...


Saving to HDF5:   0%|          | 0/28000 [00:00<?, ?it/s]


Successfully saved new dataset to /home/poorna/data/eeg_blip_only_final.h5


In [21]:
import h5py
import torch
import numpy as np
from transformers import BertTokenizer

# --- CONFIGURATION (Match paths from your generation script) ---
HDF5_FILE = "/home/poorna/data/eeg_blip_only_final.h5"
TOKENIZER_PATH = "/home/poorna/models/bert-base-uncased"
NUM_SAMPLES_TO_CHECK = 5

# --- Load Tokenizer ---
tokenizer = BertTokenizer.from_pretrained(TOKENIZER_PATH)

print(f"--- Inspecting first {NUM_SAMPLES_TO_CHECK} entries in {HDF5_FILE} ---")

try:
    with h5py.File(HDF5_FILE, "r") as f:
        print("\nDatasets available:", list(f.keys()))

        if "eeg" not in f or "blip_input_ids" not in f:
            print("Error: Required datasets ('eeg', 'blip_input_ids') not found in the HDF5 file.")
            exit()

        for i in range(NUM_SAMPLES_TO_CHECK):
            eeg_data = f["eeg"][i]
            blip_tokens = f["blip_input_ids"][i]

            # Decode the tokens back to text (ignoring special tokens like [CLS], [SEP])
            blip_caption = tokenizer.decode(
                blip_tokens, 
                skip_special_tokens=True
            )

            print(f"\n--- Sample {i} ---")
            print(f"EEG Shape: {eeg_data.shape} (62 Channels x 400 Timesteps)")
            print(f"BLIP Tokens Shape: {blip_tokens.shape}")
            print(f"Decoded BLIP Caption: **{blip_caption}**")
            
            # Optional: Print the start of the token IDs to confirm data type
            # print(f"BLIP Token IDs (start): {blip_tokens[:5]}")

except FileNotFoundError:
    print(f"\nFATAL ERROR: HDF5 file not found at {HDF5_FILE}. Please run the dataset generation script first.")
except Exception as e:
    print(f"\nAn error occurred while reading the file: {e}")

--- Inspecting first 5 entries in /home/poorna/data/eeg_blip_only_final.h5 ---

Datasets available: ['blip_input_ids', 'eeg']

--- Sample 0 ---
EEG Shape: (62, 400) (62 Channels x 400 Timesteps)
BLIP Tokens Shape: (64,)
Decoded BLIP Caption: **a city with tall buildings and a street**

--- Sample 1 ---
EEG Shape: (62, 400) (62 Channels x 400 Timesteps)
BLIP Tokens Shape: (64,)
Decoded BLIP Caption: **a city at night with buildings**

--- Sample 2 ---
EEG Shape: (62, 400) (62 Channels x 400 Timesteps)
BLIP Tokens Shape: (64,)
Decoded BLIP Caption: **a city with lots of tall buildings**

--- Sample 3 ---
EEG Shape: (62, 400) (62 Channels x 400 Timesteps)
BLIP Tokens Shape: (64,)
Decoded BLIP Caption: **a view of a city with tall buildings**

--- Sample 4 ---
EEG Shape: (62, 400) (62 Channels x 400 Timesteps)
BLIP Tokens Shape: (64,)
Decoded BLIP Caption: **a city at night with buildings lit up**
